In [3]:
!pip install mlflow
!pip install spacy
!python -m spacy download en_core_web_sm
!pip install nltk

  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached gunicorn-25.1.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached databricks_sdk-0.96.0-py3-none-any.whl.metadata (40 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached mako-1.3.10-py3-none-any.whl.metadata (2.9 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached werkzeug-3.1.6-py3-none-any.whl.metadata (4.0 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5

# Lematizacion

In [4]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import nltk
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from nltk.stem import WordNetLemmatizer
from tqdm import tqdm




tqdm.pandas()

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()


#CONFIGURACIÓN MLFLOW

mlflow.set_tracking_uri("http://ec2-3-87-48-179.compute-1.amazonaws.com:5000")

experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


#OUTPUT FOLDERS

os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


#CARGAR DATA

train_df = pd.read_csv("data/train_clean.csv")
test_df = pd.read_csv("data/test_clean.csv")

print(train_df.head())


#LEMATIZACIÓN

def clean_text(text):

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)


print("\nLematizando dataset de entrenamiento...")
train_df["text"] = train_df["text"].progress_map(clean_text)

print("\nLematizando dataset de prueba...")
test_df["text"] = test_df["text"].progress_map(clean_text)


#GUARDAR DATASET

os.makedirs("data_lem", exist_ok=True)

train_df.to_csv("data_lem/train_lem.csv", index=False)
test_df.to_csv("data_lem/test_lem.csv", index=False)


#FEATURES

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


#PIPELINE

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


#ENTRENAMIENTO

with mlflow.start_run(run_name="Lem_BOW_MultinomialNB"):

    #TIEMPO INICIAL
    start_time = time.time()

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    end_time = time.time()

    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")


    #PARAMS

    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "lemmatization",
        "dataset": "Sentiment140"
    })


    #METRICAS

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)


    #TAGS

    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")


    #LOG MODEL

    mlflow.sklearn.log_model(pipeline, "model")


    #MODEL CARD

    model_card = {
        "model_name": "NB BOW Lemmatization",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)


    #ABLATION TABLE

    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "lemmatization",
        "f1_score": f1
    }])

    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)


    #ABLATION PLOT

    plt.figure()

    plt.bar(["NB + BOW + Lem"], [f1])

    plt.ylabel("F1 Score")

    plt.title("Experiment Result")

    plt.savefig("outputs/reports/ablation_plot.png")


    #SUMMARY

    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with lemmatization preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""

    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)


    #WORK DISTRIBUTION

    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + Lemmatization"
    }])

    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)


    #LOG ARTIFACTS

    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")


    #OUTPUT

    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


#COMPARACIÓN DE RESULTADOS

comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

print(comparison_df.head())

[nltk_data] Downloading package wordnet to /home/ec2-user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ec2-user/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


                                                text  label
0  closet organizer install complete  now for the...      0
1  mornin all  i need to wake upthis will get me ...      1
2            ahhhhhhhhhh he suxxxxxxx he claimed it       0
3   haha i guess all the good bits were in the tr...      0
4                                  family guy funny       1

Lematizando dataset de entrenamiento...


100%|██████████| 1360000/1360000 [00:55<00:00, 24641.41it/s]



Lematizando dataset de prueba...


100%|██████████| 240000/240000 [00:09<00:00, 24509.80it/s]
2026/03/08 02:30:12 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 02:30:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 02:30:27 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmplg87hc1d/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 02:30:27 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series


Ejecución Exitosa
F1 Score: 0.7809710384504683
Runtime: 93.60177159309387

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.80      0.79    120129
           1       0.79      0.76      0.78    119871

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

🏃 View run Lem_BOW_MultinomialNB at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8/runs/12e02b78260c4cdeaac067d419dffd52
🧪 View experiment at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         1
2     1         1
3     1         1
4     1         1


# Stop words

In [5]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import nltk
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from nltk.corpus import stopwords
from tqdm import tqdm


tqdm.pandas()

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


#CONFIGURACIÓN MLFLOW

mlflow.set_tracking_uri("http://ec2-3-87-48-179.compute-1.amazonaws.com:5000")

experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


#OUTPUT FOLDERS

os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


#CARGAR DATA

train_df = pd.read_csv("data/train_clean.csv")
test_df = pd.read_csv("data/test_clean.csv")

print(train_df.head())


#ELIMINACIÓN DE STOPWORDS

def clean_text(text):

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)


print("\nEliminando stopwords en dataset de entrenamiento...")
train_df["text"] = train_df["text"].progress_map(clean_text)

print("\nEliminando stopwords en dataset de prueba...")
test_df["text"] = test_df["text"].progress_map(clean_text)


#GUARDAR DATASET

os.makedirs("data_stopwords", exist_ok=True)

train_df.to_csv("data_stopwords/train_stopwords.csv", index=False)
test_df.to_csv("data_stopwords/test_stopwords.csv", index=False)


#FEATURES

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


#PIPELINE

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


#ENTRENAMIENTO

with mlflow.start_run(run_name="Stopwords_BOW_MultinomialNB"):

    start_time = time.time()

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    end_time = time.time()

    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")


    #PARAMS

    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "stopwords_removal",
        "dataset": "Sentiment140"
    })


    #METRICAS

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)


    #TAGS

    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")


    #LOG MODEL

    mlflow.sklearn.log_model(pipeline, "model")


    #MODEL CARD

    model_card = {
        "model_name": "NB BOW Stopwords",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "preprocessing": "stopwords_removal",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)


    #ABLATION TABLE

    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "stopwords_removal",
        "f1_score": f1
    }])

    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)


    #ABLATION PLOT

    plt.figure()

    plt.bar(["NB + BOW + Stopwords"], [f1])

    plt.ylabel("F1 Score")

    plt.title("Experiment Result")

    plt.savefig("outputs/reports/ablation_plot.png")


    #SUMMARY

    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with stopwords removal preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""

    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)


    #WORK DISTRIBUTION

    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + Stopwords"
    }])

    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)


    #LOG ARTIFACTS

    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")


    #OUTPUT

    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


#COMPARACIÓN DE RESULTADOS

comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

print(comparison_df.head())

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                                text  label
0  closet organizer install complete  now for the...      0
1  mornin all  i need to wake upthis will get me ...      1
2            ahhhhhhhhhh he suxxxxxxx he claimed it       0
3   haha i guess all the good bits were in the tr...      0
4                                  family guy funny       1

Eliminando stopwords en dataset de entrenamiento...


100%|██████████| 1360000/1360000 [00:03<00:00, 410373.87it/s]



Eliminando stopwords en dataset de prueba...


100%|██████████| 240000/240000 [00:00<00:00, 413353.42it/s]
2026/03/08 02:38:30 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 02:38:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 02:38:41 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpwteovser/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 02:38:42 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Serie


Ejecución Exitosa
F1 Score: 0.7723332384068066
Runtime: 69.15055465698242

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.78      0.78    120129
           1       0.78      0.76      0.77    119871

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000

🏃 View run Stopwords_BOW_MultinomialNB at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8/runs/efe777ae3d2c4d6cafe5f967f39a35e5
🧪 View experiment at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         0
2     1         1
3     1         1
4     1         1


# Emojis

In [6]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import nltk
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from tqdm import tqdm

tqdm.pandas()


#CONFIGURACIÓN MLFLOW

mlflow.set_tracking_uri("http://ec2-3-87-48-179.compute-1.amazonaws.com:5000")

experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


#OUTPUT FOLDERS

os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


#CARGAR DATA

train_df = pd.read_csv("data/train_clean.csv")
test_df = pd.read_csv("data/test_clean.csv")

print(train_df.head())


#REGEX PARA EMOJIS

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport
    "\U0001F1E0-\U0001F1FF"  # flags
    "]+",
    flags=re.UNICODE
)


#REGEX PARA EMOTICONOS

emoticon_pattern = re.compile(r"(:\)|:-\)|:\(|:-\(|:D|:-D|;\)|;-\))")


#ELIMINACIÓN DE EMOJIS / EMOTICONOS

def clean_text(text):

    text = emoji_pattern.sub("", text)

    text = emoticon_pattern.sub("", text)

    return text


print("\nEliminando emojis en dataset de entrenamiento...")
train_df["text"] = train_df["text"].progress_map(clean_text)

print("\nEliminando emojis en dataset de prueba...")
test_df["text"] = test_df["text"].progress_map(clean_text)


#GUARDAR DATASET

os.makedirs("data_noemoji", exist_ok=True)

train_df.to_csv("data_noemoji/train_noemoji.csv", index=False)
test_df.to_csv("data_noemoji/test_noemoji.csv", index=False)


#FEATURES

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


#PIPELINE

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


#ENTRENAMIENTO

with mlflow.start_run(run_name="EmojiDrop_BOW_MultinomialNB"):

    start_time = time.time()

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    end_time = time.time()

    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")


    #PARAMS

    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "emoji_drop",
        "dataset": "Sentiment140"
    })


    #METRICAS

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)


    #TAGS

    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")


    #LOG MODEL

    mlflow.sklearn.log_model(pipeline, "model")


    #MODEL CARD

    model_card = {
        "model_name": "NB BOW Emoji Removal",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "preprocessing": "emoji_removal",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)


    #ABLATION TABLE

    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "emoji_removal",
        "f1_score": f1
    }])

    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)


    #ABLATION PLOT

    plt.figure()

    plt.bar(["NB + BOW + EmojiDrop"], [f1])

    plt.ylabel("F1 Score")

    plt.title("Experiment Result")

    plt.savefig("outputs/reports/ablation_plot.png")


    #SUMMARY

    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with emoji removal preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""

    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)


    #WORK DISTRIBUTION

    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + Emoji Removal"
    }])

    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)


    #LOG ARTIFACTS

    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")


    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


#COMPARACIÓN DE RESULTADOS

comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

print(comparison_df.head())

                                                text  label
0  closet organizer install complete  now for the...      0
1  mornin all  i need to wake upthis will get me ...      1
2            ahhhhhhhhhh he suxxxxxxx he claimed it       0
3   haha i guess all the good bits were in the tr...      0
4                                  family guy funny       1

Eliminando emojis en dataset de entrenamiento...


100%|██████████| 1360000/1360000 [00:03<00:00, 374632.90it/s]



Eliminando emojis en dataset de prueba...


100%|██████████| 240000/240000 [00:00<00:00, 374986.53it/s]
2026/03/08 02:41:45 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 02:41:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 02:42:00 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp8zqoraz8/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 02:42:01 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Serie


Ejecución Exitosa
F1 Score: 0.7824890326122277
Runtime: 95.11618900299072

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.80      0.79    120129
           1       0.79      0.76      0.78    119871

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

🏃 View run EmojiDrop_BOW_MultinomialNB at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8/runs/3092e5e7087d49bc922ee6940140b476
🧪 View experiment at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         1
2     1         1
3     1         1
4     1         1


# Puntuacion

In [7]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from tqdm import tqdm

tqdm.pandas()


# CONFIGURACIÓN MLFLOW

mlflow.set_tracking_uri("http://ec2-3-87-48-179.compute-1.amazonaws.com:5000")

experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


# OUTPUT FOLDERS

os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


# CARGAR DATA

train_df = pd.read_csv("data/train_clean.csv")
test_df = pd.read_csv("data/test_clean.csv")

print(train_df.head())


# REGEX PARA PUNTUACIÓN Y SÍMBOLOS

punctuation_pattern = re.compile(r"[^\w\s]")


# LIMPIEZA: ELIMINAR PUNTUACIÓN / SÍMBOLOS

def clean_text(text):

    text = punctuation_pattern.sub("", text)

    return text


print("\nEliminando puntuación y símbolos en dataset de entrenamiento...")
train_df["text"] = train_df["text"].progress_map(clean_text)

print("\nEliminando puntuación y símbolos en dataset de prueba...")
test_df["text"] = test_df["text"].progress_map(clean_text)


# GUARDAR DATASET

os.makedirs("data_nopunct", exist_ok=True)

train_df.to_csv("data_nopunct/train_nopunct.csv", index=False)
test_df.to_csv("data_nopunct/test_nopunct.csv", index=False)


# FEATURES

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


# PIPELINE

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


# ENTRENAMIENTO

with mlflow.start_run(run_name="PunctuationDrop_BOW_MultinomialNB"):

    start_time = time.time()

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    end_time = time.time()

    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")


    # PARAMS

    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "punctuation_drop",
        "dataset": "Sentiment140"
    })


    # MÉTRICAS

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)


    # TAGS

    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")


    # LOG MODEL

    mlflow.sklearn.log_model(pipeline, "model")


    # MODEL CARD

    model_card = {
        "model_name": "NB BOW Punctuation Removal",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "preprocessing": "punctuation_removal",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)


    # ABLATION TABLE

    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "punctuation_removal",
        "f1_score": f1
    }])

    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)


    # ABLATION PLOT

    plt.figure()

    plt.bar(["NB + BOW + NoPunct"], [f1])

    plt.ylabel("F1 Score")
    plt.title("Experiment Result")

    plt.savefig("outputs/reports/ablation_plot.png")


    # SUMMARY

    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with punctuation and symbol removal preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""

    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)


    # WORK DISTRIBUTION

    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + Punctuation Removal"
    }])

    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)


    # LOG ARTIFACTS

    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")


    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


# COMPARACIÓN DE RESULTADOS

comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

print(comparison_df.head())

                                                text  label
0  closet organizer install complete  now for the...      0
1  mornin all  i need to wake upthis will get me ...      1
2            ahhhhhhhhhh he suxxxxxxx he claimed it       0
3   haha i guess all the good bits were in the tr...      0
4                                  family guy funny       1

Eliminando puntuación y símbolos en dataset de entrenamiento...


100%|██████████| 1360000/1360000 [00:02<00:00, 673894.82it/s]



Eliminando puntuación y símbolos en dataset de prueba...


100%|██████████| 240000/240000 [00:00<00:00, 671491.53it/s]
2026/03/08 02:47:25 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 02:47:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 02:47:40 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmph4yvxp5z/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 02:47:41 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Serie


Ejecución Exitosa
F1 Score: 0.7824890326122277
Runtime: 94.80471134185791

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.80      0.79    120129
           1       0.79      0.76      0.78    119871

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

🏃 View run PunctuationDrop_BOW_MultinomialNB at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8/runs/7e4eda5362254b43adb0bf3b71d27fd1
🧪 View experiment at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         1
2     1         1
3     1         1
4     1         1


# Alargamientos

In [8]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from tqdm import tqdm

tqdm.pandas()


# CONFIGURACIÓN MLFLOW

mlflow.set_tracking_uri("http://ec2-3-87-48-179.compute-1.amazonaws.com:5000")

experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


# OUTPUT FOLDERS

os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


# CARGAR DATA

train_df = pd.read_csv("data/train_clean.csv")
test_df = pd.read_csv("data/test_clean.csv")

print(train_df.head())


# REGEX PARA NORMALIZAR ALARGAMIENTOS
# reduce letras repetidas (3+) a solo 2

elongation_pattern = re.compile(r"(.)\1{2,}")


def normalize_elongation(text):

    return elongation_pattern.sub(r"\1\1", text)


print("\nNormalizando alargamientos en dataset de entrenamiento...")
train_df["text"] = train_df["text"].progress_map(normalize_elongation)

print("\nNormalizando alargamientos en dataset de prueba...")
test_df["text"] = test_df["text"].progress_map(normalize_elongation)


# GUARDAR DATASET

os.makedirs("data_elongation_norm", exist_ok=True)

train_df.to_csv("data_elongation_norm/train_elongation.csv", index=False)
test_df.to_csv("data_elongation_norm/test_elongation.csv", index=False)


# FEATURES

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


# PIPELINE

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


# ENTRENAMIENTO

with mlflow.start_run(run_name="ElongationNorm_BOW_MultinomialNB"):

    start_time = time.time()

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    end_time = time.time()

    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")


    # PARAMS

    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "elongation_normalization",
        "dataset": "Sentiment140"
    })


    # MÉTRICAS

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)


    # TAGS

    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")


    # LOG MODEL

    mlflow.sklearn.log_model(pipeline, "model")


    # MODEL CARD

    model_card = {
        "model_name": "NB BOW Elongation Normalization",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "preprocessing": "elongation_normalization",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)


    # ABLATION TABLE

    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "elongation_normalization",
        "f1_score": f1
    }])

    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)


    # ABLATION PLOT

    plt.figure()

    plt.bar(["NB + BOW + ElongNorm"], [f1])

    plt.ylabel("F1 Score")
    plt.title("Experiment Result")

    plt.savefig("outputs/reports/ablation_plot.png")


    # SUMMARY

    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with elongation normalization preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""

    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)


    # WORK DISTRIBUTION

    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + Elongation Normalization"
    }])

    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)


    # LOG ARTIFACTS

    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")


    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


# COMPARACIÓN DE RESULTADOS

comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

print(comparison_df.head())

                                                text  label
0  closet organizer install complete  now for the...      0
1  mornin all  i need to wake upthis will get me ...      1
2            ahhhhhhhhhh he suxxxxxxx he claimed it       0
3   haha i guess all the good bits were in the tr...      0
4                                  family guy funny       1

Normalizando alargamientos en dataset de entrenamiento...


100%|██████████| 1360000/1360000 [00:04<00:00, 328314.20it/s]



Normalizando alargamientos en dataset de prueba...


100%|██████████| 240000/240000 [00:00<00:00, 325646.93it/s]
2026/03/08 02:52:28 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 02:52:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 02:52:44 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpjjnsz31_/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 02:52:44 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Serie


Ejecución Exitosa
F1 Score: 0.7829392714882779
Runtime: 95.4617371559143

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.80      0.79    120129
           1       0.79      0.76      0.78    119871

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

🏃 View run ElongationNorm_BOW_MultinomialNB at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8/runs/389887e6de3b4d8e97b3d7d34097763c
🧪 View experiment at: http://ec2-3-87-48-179.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         1
2     1         1
3     1         1
4     1         1


# Puntuacion + Alargamientos

In [12]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from tqdm import tqdm

tqdm.pandas()


# CONFIGURACIÓN MLFLOW

mlflow.set_tracking_uri("http://ec2-54-157-225-114.compute-1.amazonaws.com:5000")

experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


# OUTPUT FOLDERS

os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


# CARGAR DATA (DATASET SIN PUNTUACIÓN)

train_df = pd.read_csv("data_nopunct/train_nopunct.csv")
test_df = pd.read_csv("data_nopunct/test_nopunct.csv")

print(train_df.head())


# REGEX PARA NORMALIZAR ALARGAMIENTOS

elongation_pattern = re.compile(r"(.)\1{2,}")


def normalize_elongation(text):

    return elongation_pattern.sub(r"\1\1", text)


print("\nNormalizando alargamientos en dataset de entrenamiento...")
train_df["text"] = train_df["text"].progress_map(normalize_elongation)

print("\nNormalizando alargamientos en dataset de prueba...")
test_df["text"] = test_df["text"].progress_map(normalize_elongation)


# GUARDAR DATASET FINAL

os.makedirs("data_nopunct_elongation", exist_ok=True)

train_df.to_csv("data_nopunct_elongation/train_final.csv", index=False)
test_df.to_csv("data_nopunct_elongation/test_final.csv", index=False)


# FEATURES

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


# PIPELINE

pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


# ENTRENAMIENTO

with mlflow.start_run(run_name="NoPunct_ElongNorm_BOW_MultinomialNB"):

    start_time = time.time()

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    end_time = time.time()

    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")


    # PARAMS

    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "punctuation_removal + elongation_normalization",
        "dataset": "Sentiment140"
    })


    # MÉTRICAS

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)


    # TAGS

    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "ablation")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")


    # LOG MODEL

    mlflow.sklearn.log_model(pipeline, "model")


    # MODEL CARD

    model_card = {
        "model_name": "NB BOW NoPunct + ElongationNorm",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "preprocessing": "punctuation_removal + elongation_normalization",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)


    # ABLATION TABLE

    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "NoPunct + ElongNorm",
        "f1_score": f1
    }])

    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)


    # ABLATION PLOT

    plt.figure()

    plt.bar(["NB + BOW + NoPunct + ElongNorm"], [f1])

    plt.ylabel("F1 Score")
    plt.title("Experiment Result")

    plt.savefig("outputs/reports/ablation_plot.png")


    # SUMMARY

    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with punctuation removal + elongation normalization preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""

    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)


    # WORK DISTRIBUTION

    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + NoPunct + ElongNorm"
    }])

    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)


    # LOG ARTIFACTS

    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")


    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


# COMPARACIÓN DE RESULTADOS

comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})

print(comparison_df.head())

                                                text  label
0  closet organizer install complete  now for the...      0
1  mornin all  i need to wake upthis will get me ...      1
2            ahhhhhhhhhh he suxxxxxxx he claimed it       0
3   haha i guess all the good bits were in the tr...      0
4                                  family guy funny       1

Normalizando alargamientos en dataset de entrenamiento...


100%|██████████| 1360000/1360000 [00:04<00:00, 333886.57it/s]



Normalizando alargamientos en dataset de prueba...


100%|██████████| 240000/240000 [00:00<00:00, 332862.89it/s]
2026/03/08 03:29:04 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 03:29:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 03:29:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpepkwgegl/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 03:29:20 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Serie


Ejecución Exitosa
F1 Score: 0.7829392714882779
Runtime: 94.39021110534668

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.80      0.79    120129
           1       0.79      0.76      0.78    119871

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

🏃 View run NoPunct_ElongNorm_BOW_MultinomialNB at: http://ec2-54-157-225-114.compute-1.amazonaws.com:5000/#/experiments/8/runs/c7c27fac475a4244bd5194f24ad76a04
🧪 View experiment at: http://ec2-54-157-225-114.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         1
2     1         1
3     1         1
4     1         1


# Stop words + Puntuacion + Alargamientos

In [15]:
import re
import numpy as np
import pandas as pd
import mlflow
import os
import time
import json
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

from tqdm import tqdm
tqdm.pandas()


#CONFIGURACIÓN MLFLOW
mlflow.set_tracking_uri("http://ec2-54-157-225-114.compute-1.amazonaws.com:5000")
experiment_name = "Parcial_NLP"
mlflow.set_experiment(experiment_name)


#OUTPUT FOLDERS
os.makedirs("outputs/model", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


#CARGAR DATASET LIMPIO DE STOPWORDS
train_df = pd.read_csv("data_stopwords/train_stopwords.csv")
test_df = pd.read_csv("data_stopwords/test_stopwords.csv")

print(train_df.head())

# Rellenar NaN para evitar errores
train_df['text'] = train_df['text'].fillna("")
test_df['text'] = test_df['text'].fillna("")


#REGEX PARA PUNTUACIÓN Y NORMALIZAR ALARGAMIENTOS
punctuation_pattern = re.compile(r"[^\w\s]")
elongation_pattern = re.compile(r"(.)\1{2,}")  # reduce letras repetidas 3+ a 2


#FUNCION DE LIMPIEZA: PUNCTUATION + ELONGATION
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Eliminar puntuación
    text = punctuation_pattern.sub("", text)
    # Normalizar alargamientos
    text = elongation_pattern.sub(r"\1\1", text)
    return text


#APLICAR LIMPIEZA
print("\nLimpiando dataset de entrenamiento (Stopwords + NoPunct + ElongationNorm)...")
train_df["text"] = train_df["text"].progress_map(clean_text)

print("\nLimpiando dataset de prueba (Stopwords + NoPunct + ElongationNorm)...")
test_df["text"] = test_df["text"].progress_map(clean_text)


#GUARDAR DATASET PROCESADO
os.makedirs("data_stop+elong+punct", exist_ok=True)
train_df.to_csv("data_stop+elong+punct/train_final.csv", index=False)
test_df.to_csv("data_stop+elong+punct/test_final.csv", index=False)


#FEATURES
X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]


#PIPELINE
pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB(alpha=1.0))
])


#ENTRENAMIENTO
with mlflow.start_run(run_name="Stop_NoPunct_ElongNorm_BOW_MultinomialNB"):

    start_time = time.time()
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    end_time = time.time()
    runtime = end_time - start_time

    f1 = f1_score(y_test, y_pred, average="weighted")

    # PARAMS
    mlflow.log_params({
        "alpha": 1.0,
        "preprocessing": "stopwords + punctuation_removal + elongation_normalization",
        "dataset": "Sentiment140"
    })

    # MÉTRICAS
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("runtime_seconds", runtime)

    # TAGS
    mlflow.set_tag("author", "Nicolas")
    mlflow.set_tag("experiment_type", "ablation")
    mlflow.set_tag("feature_type", "BOW")
    mlflow.set_tag("dataset", "Sentiment140")

    # LOG MODEL
    mlflow.sklearn.log_model(pipeline, "model")

    # MODEL CARD
    model_card = {
        "model_name": "NB BOW Stopwords + NoPunct + ElongNorm",
        "vectorizer": "CountVectorizer",
        "classifier": "MultinomialNB",
        "dataset": "Sentiment140",
        "preprocessing": "stopwords + punctuation_removal + elongation_normalization",
        "f1_score": float(f1),
        "author": "Nicolas"
    }

    with open("outputs/model/model_card.json", "w") as f:
        json.dump(model_card, f, indent=4)

    # ABLATION TABLE
    ablation_df = pd.DataFrame([{
        "vectorizer": "CountVectorizer",
        "model": "MultinomialNB",
        "preprocessing": "Stopwords + NoPunct + ElongNorm",
        "f1_score": f1
    }])
    ablation_df.to_csv("outputs/reports/ablation_results.csv", index=False)

    # ABLATION PLOT
    plt.figure()
    plt.bar(["NB + BOW + Stop + NoPunct + ElongNorm"], [f1])
    plt.ylabel("F1 Score")
    plt.title("Experiment Result")
    plt.savefig("outputs/reports/ablation_plot.png")

    # SUMMARY
    summary = f"""
Experiment using Bag of Words and Multinomial Naive Bayes
with stopwords removal + punctuation removal + elongation normalization preprocessing.

F1-score achieved: {f1:.4f}

Dataset: Sentiment140
"""
    with open("outputs/reports/ablation_summary.txt", "w") as f:
        f.write(summary)

    # WORK DISTRIBUTION
    work_df = pd.DataFrame([{
        "member": "Nicolas",
        "experiment": "NB + BOW + Stop + NoPunct + ElongNorm"
    }])
    work_df.to_csv("outputs/reports/work_distribution.csv", index=False)

    # LOG ARTIFACTS
    mlflow.log_artifact("outputs/model/model_card.json")
    mlflow.log_artifact("outputs/reports/ablation_results.csv")
    mlflow.log_artifact("outputs/reports/ablation_plot.png")
    mlflow.log_artifact("outputs/reports/ablation_summary.txt")
    mlflow.log_artifact("outputs/reports/work_distribution.csv")

    # OUTPUT
    print("\nEjecución Exitosa")
    print("F1 Score:", f1)
    print("Runtime:", runtime)
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))


#COMPARACIÓN DE RESULTADOS
comparison_df = pd.DataFrame({
    "Real": y_test,
    "Predicho": y_pred
})
print(comparison_df.head())

                                                text  label
0  closet organizer install complete torture part...      0
1                   mornin need wake upthis get goin      1
2                      ahhhhhhhhhh suxxxxxxx claimed      0
3                       haha guess good bits trailer      0
4                                   family guy funny      1

Limpiando dataset de entrenamiento (Stopwords + NoPunct + ElongationNorm)...


100%|██████████| 1360000/1360000 [00:04<00:00, 297829.18it/s]



Limpiando dataset de prueba (Stopwords + NoPunct + ElongationNorm)...


100%|██████████| 240000/240000 [00:00<00:00, 299436.12it/s]
2026/03/08 03:43:11 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/08 03:43:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/08 03:43:23 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp_ppghwe4/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/03/08 03:43:23 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Serie


Ejecución Exitosa
F1 Score: 0.7729144575221335
Runtime: 69.83429837226868

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.78      0.78    120129
           1       0.78      0.76      0.77    119871

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000

🏃 View run Stop_NoPunct_ElongNorm_BOW_MultinomialNB at: http://ec2-54-157-225-114.compute-1.amazonaws.com:5000/#/experiments/8/runs/ffc0a78e87134663b3c3c72987f05ca3
🧪 View experiment at: http://ec2-54-157-225-114.compute-1.amazonaws.com:5000/#/experiments/8
   Real  Predicho
0     1         1
1     1         0
2     1         1
3     1         1
4     1         1
